## Notebook47

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs
from funs import DSImage
from PIL import Image
import cv2

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
fsac = pl.read_csv(ub + "data/fsac.csv")

Unlike the CSV files in earlier notebooks, an image dataset needs the image
files themselves on disk before `cv2.imread` can open them. This cell
downloads the photographs referenced in `fsac` the first time you run it and
does nothing on later runs (or if you already have them).

In [ ]:
import os
import urllib.request

for row in fsac.iter_rows(named=True):
    if not os.path.exists(row["filepath"]):
        os.makedirs(os.path.dirname(row["filepath"]), exist_ok=True)
        urllib.request.urlretrieve(ub + row["filepath"], row["filepath"])

### Questions

`fsac` holds 500 color photographs from the Farm Security Administration / Office of
War Information collection at the Library of Congress, taken in the late 1930s and
early 1940s. Each row is one photograph. `filepath` points to the image file, and
columns such as `photographer`, `caption`, `year`, and `state` describe it.

Unless a question says otherwise, just print the result of each block; do not save
it to a variable.

1. Count the number of photographs taken by each `photographer`. Sort from most to
fewest photographs.

Jack Delano took almost a third of the photographs (162 of 500). Six photographers
have at least 20 photographs each.

2. Count the number of photographs taken in each `year`. Sort by `year`.

3. Take the code from the previous question and draw it as a bar plot with
`geom_col`, with `year` on the x-axis and the count on the y-axis.

Most of the photographs were taken in 1942 and 1943, during the Second World War.

4. Show the first 12 photographs in a grid. Take `fsac`, keep the first 12 rows,
and pipe the result into `DSImage.plot_image_grid` with `label_name="photographer"`
and `ncol=4`.

5. Read the first image with `cv2.imread(fsac.select(c.filepath).row(0)[0])` and
print its `.shape`.

The image is 513 pixels tall and 640 pixels wide, with three color channels. OpenCV
stores the channels in blue-green-red order.

6. Do the same as the previous question, but print the image's `.mean()` instead of
its shape.

The mean is about 59.6 on a scale from 0 to 255, so this is a fairly dark image.

7. Following the chapter, loop over every row of `fsac`. For each photograph, read
the image with `cv2.imread`, append its `.mean()` to a list called `brightness`,
convert it to HSV with `cv2.cvtColor(img, cv2.COLOR_BGR2HSV)`, and append the
output of `DSImage.compute_colors()` to a list called `colors`. Then combine
`fsac` with a `brightness` column and `pl.DataFrame(colors)` using
`pl.concat(..., how="horizontal")`. Save the result as `fsac_img`, a new name, so
that `fsac` does not change and the block is safe to run more than once. The rest of
the notebook uses `fsac_img`.

8. Sort `fsac_img` by `brightness` and show the 12 darkest photographs in a grid,
labeled by `photographer`.

The darkest photographs are night scenes and interiors, such as rail yards at night
and factory floors. A large dark background pulls the mean down no matter what the
subject is. The first two tiles show the same scene, because the collection has a
few near-duplicate frames.

9. Draw a boxplot of `brightness` by `photographer`, with `coord_flip()`. Only
include the photographers with at least 20 photographs by starting with
`.filter(pl.len().over(c.photographer) >= 20)`, and order the photographers by their
mean brightness with `.sort(c.brightness.mean().over(c.photographer))`, as in the
chapter.

John Collier and Jack Delano have the brightest photographs on average. Alfred
Palmer and Howard Hollem, who mostly worked inside factories and defense plants,
have the darkest.

10. Show the 12 photographs with the largest value of `blue` in a grid, labeled by
`state`.

Almost all of these photographs have a large area of sky. The `blue` column measures
the background as much as the subject.

11. Change the code from question 9 to draw a boxplot of `blue` by `photographer`
instead of `brightness`.

For most photographers the median is close to zero, because most of their
photographs show little or no sky. John Collier's are the most blue, and Marion Post
Wolcott's have almost no blue at all.